# Tournament analysis (260723): which models win in competition

Analyses the `260723` tournament in `~/Unity/Octagon/simulations/tournaments/260723`, run by `agent_training/tournament/run_tournament_260723.ipynb`. 8 models, every ordered pair with mirrors plus self-matchups = **64 matchups**. Folder `260723_00A__vs__260723_00B` means model A drives agent 0 (P1), model B drives agent 1 (P2). Each trial is winner-take-all: the winner scores **1.0** (reached High) or **0.4** (reached Low), the loser 0.

All extraction, statistics and figures come from `analysis/tournament_analysis.py`, shared with the `260715` notebooks; this notebook holds the config and the written-up results.

**This notebook covers**
1. **Run validation** — are the matchups complete and fair (self-matchups near 50%, mirror-consistent)?
2. **Per-run results + head-to-head matrices** — the outcome of each matchup; raw score and win-rate matrices.
3. **Overall ranking** — mean score per trial, total accumulated score, win rate, and a Bradley-Terry strength, per model.
4. **Pair interaction** — how much each head-to-head deviates from what the models' overall strengths predict (Bradley-Terry residual), plus a non-transitivity check for rock-paper-scissors loops.
5. **Reward** — the same ranking on reward instead of score, and whether efficiency reorders the models.

> **Build note — and why section 5 is worth reading for this batch.** The `260723` models were trained on `build_260723_01_social`, with the no-movement penalty **break removed**: every step is penalised, resting included. The tournament build (`build_260724_01_tournament`) has the break absent too, so here the tournament reward structure **matches the training reward structure**. For `260715` it did not (trained with the break present), which is why `tournament_analysis_demo_2` had to be held back. The open question this batch answers: with resting no longer free, does the central/peripheral ranking from `260715` reproduce?

> **Data note.** Run after the tournament finishes (every matchup has `DONE.txt`). The validation section will flag any incomplete or unfair matchups.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import analysis.tournament_analysis as ta

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

BASE = os.path.expanduser('~/Unity/Octagon/simulations/tournaments/260723')

# entrants and their strategy band, from the Stage-4 shortlist in
# demos/strategy_characterisation/competitive/model_characterisation_demo260723.ipynb.
# Dict order sets the row/column order of every matrix and the legend order, so keep the
# models ordered within a band the way you want them read.
STRATEGY = {34: 'central', 39: 'central',
            44: 'intermediate',
            35: 'peripheral', 1: 'peripheral', 33: 'peripheral',
            51: 'peripheral', 23: 'peripheral'}
MODELS = ta.models_in_order(STRATEGY)      # grouped by band, in the order above

STEP_PENALTY = 0.001    # for reference; reward = score - STEP_PENALTY * action_steps

## Main pass — load every matchup

Loads each of the 64 matchups (~5 s each), extracts per-trial outcomes, and stacks them into one long table. Also reports which matchups are missing `DONE.txt`.

In [ ]:
# One row per completed trial per matchup: both models, the winning client, each model's
# score and reward, and the wall separation.
trials_df = ta.load_tournament(BASE)
per_matchup = ta.per_matchup_summary(trials_df)

# One row per (focus model, opponent) per trial, so both seat orders pool. Self-matchups
# are dropped — a model cannot out-rank itself. Every trial appears twice, once framed from
# each model's side, so ALWAYS index this table by `focus`.
selfless = ta.perspective_table(trials_df)

## 1. Run validation

Before trusting any ranking, confirm the tournament is fair:
- **Self-matchups** (a model vs a copy of itself) should sit near **0.5** — far from it means a seat/agent asymmetry (the failure mode of the first `260715` run, where the opponent never moved).
- **Mirror consistency** — a model's win rate against an opponent should be the same whether it played P1 or P2. Large gaps mean the seat still matters. `260715` came out at a mean gap of 0.018.
- **Winner-take-all** — exactly one agent scores each trial.

Score is safe to use here even though the build's reward structure is in question, because these are self- and mirror-matchups only, where score proportions cannot differ by construction.

In [ ]:
ta.validation_report(trials_df, per_matchup, MODELS);

## 2. Per-run results and head-to-head

Each matchup's outcome, then the pooled head-to-head matrices. Pooling both seat orders (A-vs-B and B-vs-A) gives a seat-balanced estimate of how each model does against each opponent.

In [ ]:
# Mirror-pooled matrices, read row-model against column-opponent.
mats = ta.head_to_head(selfless, MODELS)
ta.plot_head_to_head(mats)
plt.show()

print("Per-matchup results (ordered A=P1 vs B=P2):")
per_matchup.round(3)

## 3. Overall ranking

Which models are most successful, over all opponents (self-matchups excluded). The focus is **mean score per trial**; total accumulated score and win rate are also shown, plus a Bradley-Terry strength fit from the win/loss counts.

The comparison to make against `260715`: there, all three central models took the top three places and all three peripheral models the bottom three, with no overlap. If the penalty break was what made central play pay, that separation should shrink here.

In [ ]:
overall = ta.overall_ranking(selfless, STRATEGY, MODELS, sort_by='mean_score')
ta.plot_overall_ranking(overall, metric='mean_score')
plt.show()

## 4. Does the pairing matter? (interaction)

If success were purely each model's general strength, every head-to-head would be predicted by the two Bradley-Terry strengths. The **residual** (actual win rate minus BT-predicted) is the part explained by the specific pairing — the interaction. Large residuals, or non-transitive loops (A beats B beats C beats A), mean the matchup itself matters, not just overall strength. `260715`: RMS residual 0.034, zero loops.

In [ ]:
bt = ta.bradley_terry(selfless, MODELS)
resid = ta.interaction(mats['win'], bt, MODELS, strategy=STRATEGY)
ta.plot_interaction(resid)
plt.show()

## 5. Reward — does efficiency reorder the models?

- **score** = terminal outcome (winner 1.0 High / 0.4 Low, loser 0).
- **reward** (`trialRewards`) = score − step_penalty × episode steps.

For this batch the tournament build's reward structure matches the training build's, so unlike `260715` this ranking reflects the landscape the models were actually optimised under. Episode length is still pinned by the predetermined trial sequence and symmetric between the two competitors, so if the per-model movement cost comes out near-constant again, reward will still add nothing over score — but that is now a **result** about these models rather than an artefact of the wrong build.

In [ ]:
ta.plot_reward_head_to_head(mats)
plt.show()

# Same table as above, ranked on reward instead. Wins (and so the Bradley-Terry strengths)
# are identical either way — reward changes the value of a trial, not its winner.
overall_reward = ta.overall_ranking(selfless, STRATEGY, MODELS, sort_by='mean_reward')
ta.plot_overall_ranking(overall_reward, metric='mean_reward', with_bt=False)
plt.show()

In [ ]:
cmp = ta.score_vs_reward(overall)
ta.plot_score_vs_reward(cmp)
plt.show()

## Reading the results

- **Validation** — trust the rest only if self-matchups sit near 0.5 and mirror gaps are small.
- **Overall ranking** — mean score per trial; BT strength should track it closely. Colour shows whether a strategy band dominates. Compare the band separation directly against `tournament_analysis_demo` (260715).
- **Interaction** — a small RMS residual and no non-transitive loops means success is essentially each model's general strength, and the pairing adds little. Large residuals or loops mean specific matchups (and strategy clashes) matter beyond overall strength.
- **Reward** — a `rank_shift` of 0 for every model means the movement cost is a constant across models and reward is score minus that constant. A non-zero shift is the interesting outcome: it would mean cheap play genuinely buys rank once resting is no longer free.
- Score is winner-take-all, so a model's mean score combines how often it wins and whether it wins High vs Low; the win-rate matrix separates those.

The band composition differs from `260715` (2 central / 1 intermediate / 5 peripheral here, against 3/2/3 there) because the characterisation shortlist put more `260723` models on the peripheral side. Band *means* are therefore less comparable across the two batches than the per-model ordering is.

Situational analysis (per wall separation, using the `sep` column) and behavioural strategy characterisation are deliberately left to separate notebooks.